# Predict What Happens Next — AI4I 2020 Predictive Maintenance
**Problem ID: ALG-DATA-02** · Binary classification: will the machine fail?

Plan: EDA/cleaning → feature engineering → model selection with stratified CV → threshold tuning → unseen predictions

In [ ]:
import pandas as pd, numpy as np, glob
df = pd.read_csv(glob.glob('data/*.csv')[0])
df.head(), df.shape, df['Machine failure'].value_counts(normalize=True)

### 1. Key insight: extreme class imbalance (~3.4% failures)
- Accuracy is useless; **PR-AUC, F1 and Recall** are the honest metrics
- Failure-type columns (TWF, HDF, PWF, OSF, RNF) are derived from the target → **drop them (leakage)**

In [ ]:
df = df.drop(columns=['TWF','HDF','PWF','OSF','RNF']).drop_duplicates()
df[['Air temperature [K]','Torque [Nm]','Tool wear [min]','Machine failure']].describe()

### 2. Physics-motivated features (this is your 'feature reasoning' story)
| Feature | Why |
|---|---|
| temp_delta / temp_ratio | Overheating drives HDF failures |
| power = torque × rotational speed | Overload drives PWF/OSF |
| torque_wear, power_wear | Worn tool under high load is dangerous |
| torque_dev | Deviation from nominal 340 Nm |

In [ ]:
def engineer(df):
    df = df.copy()
    df['type_code'] = df['Type'].astype('category').cat.codes
    df['temp_delta'] = df['Process temperature [K]'] - df['Air temperature [K]']
    df['temp_ratio'] = df['Process temperature [K]'] / df['Air temperature [K]']
    df['power'] = df['Torque [Nm]'] * df['Rotational speed [rpm]'] * 2*np.pi/60
    df['torque_wear'] = df['Torque [Nm]'] * df['Tool wear [min]']
    df['power_wear'] = df['power'] * np.sqrt(df['Tool wear [min]'] + 1)
    df['torque_dev'] = np.abs(df['Torque [Nm]'] - 340.0)
    return df
df = engineer(df)

### 3+4. Model selection — 5-fold Stratified CV, decision threshold tuned on OOF predictions

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
FEATS = ['type_code','Air temperature [K]','Process temperature [K]','Rotational speed [rpm]',
         'Torque [Nm]','Tool wear [min]','temp_delta','temp_ratio','power','torque_wear','power_wear','torque_dev']
X, y = df[FEATS], df['Machine failure']
skf = StratifiedKFold(5, shuffle=True, random_state=42)
model = xgb.XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=5,
    scale_pos_weight=(y==0).sum()/(y==1).sum(), eval_metric='aucpr', random_state=42)
oof = np.zeros(len(y))
for tr, va in skf.split(X, y):
    model.fit(X.iloc[tr], y.iloc[tr])
    oof[va] = model.predict_proba(X.iloc[va])[:,1]
print('PR-AUC:', average_precision_score(y, oof), '| ROC-AUC:', roc_auc_score(y, oof))

### 5. Metrics + 6. Unseen predictions
See `pipeline.py` for the full leaderboard, confusion matrices, final training on all data, and unseen prediction export.